# Estimating flood damage
**In this notebook a model is created which is able to detect houses area which was heavily affected by floods and mud floods during the late spring 2023 at the shoreline of Lake Kivu.**

Simon Rüegg · SDS320 · December 2026

---



## 1. Set up

Setup instructions and the required conda environment are described in the [README](README.md).\
To run the notebook without get the kernel crashing, following cell must be run so that GDAL and PROJ work as they should. 

In [20]:
import os, sys
env = sys.prefix
os.environ["PATH"] = os.path.join(env, "Library", "bin") + os.pathsep + os.environ["PATH"]
os.environ["GDAL_DATA"] = os.path.join(env, "Library", "share", "gdal")
os.environ["PROJ_DATA"] = os.path.join(env, "Library", "share", "proj")

In a next step all libraries in this notebook need ti be loaded. Also functions to merge and clip raster files later are loaded in this step

In [21]:
# Third-party libraries
import ee
import geemap
import geoai
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import shapely
from affine import Affine
from dotenv import load_dotenv
from matplotlib.patches import Patch
from rasterio.enums import Resampling
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.windows import Window, from_bounds
from scipy.interpolate import RBFInterpolator
from shapely.geometry import Polygon, mapping

# Local modules
from functions.clip_tifs import clip_tifs
from functions.merge_tifs import merge_tifs

---
## 2. Data



### 2.1 Satellite imagery data

The data used for this project is from the Vantor's Open Data Program. Vantor provides there data from pre- and post disasters. To download the data a Vantor account is needed. This notebook works with the visual images. Vantor would also provide multispectral and panchromatic images from the same area and times.

In [3]:
# load the rasters
# the pre event tifs
rgb_pre_north = "../data/raw/raster/10300100E57BC400-visual_pre_north.tif" # pre event
rgb_pre_south = "../data/raw/raster/10300100E57BC400-visual_pre_south.tif" # pre event
rgb_post_north = "../data/raw/raster/1040010084614200-visual_post_north.tif" # post event
rgb_post_south = "../data/raw/raster/1040010084614200-visual_post_south.tif" # post event

Since the files not come as one file but in parts the different files need to be merged together.

In [4]:
# merge the files
rgb_pre = merge_tifs([rgb_pre_north, rgb_pre_south], "../data/processed/rgb_pre.tif") # pre event
rgb_post = merge_tifs([rgb_post_north, rgb_post_south], "../data/processed/rgb_post.tif") # post event

Skipping (already exists): ..\data\processed\rgb_pre.tif
Skipping (already exists): ..\data\processed\rgb_post.tif


Following code can be run to inspect the loaded and then merged rasters. It is important that the two rasters have the same CRS, bands and the same resolution to make the results meaningful.

In [7]:
with rasterio.open("../data/processed/rgb_pre.tif") as src:
    print("Visual RGB pre-event")
    print("--------------------")
    print(f"  {'Size (W × H)':<14} {src.width:,} × {src.height:,} px")
    print(f"  {'Bands':<14} {src.count}")
    print(f"  {'Data type':<14} {src.dtypes[0]}")
    print(f"  {'CRS':<14} {src.crs}")
    print(f"  {'Resolution':<14} {src.res[0]:.3f}, {src.res[1]:.3f}")
    print(f"  {'Bounds':<14} {src.bounds}")
    print(f"  {'NoData':<14} {src.nodata}")
    print()

with rasterio.open("../data/processed/rgb_post.tif") as src:
    print("Visual RGB post-event")
    print("---------------------")
    print(f"  {'Size (W × H)':<14} {src.width:,} × {src.height:,} px")
    print(f"  {'Bands':<14} {src.count}")
    print(f"  {'Data type':<14} {src.dtypes[0]}")
    print(f"  {'CRS':<14} {src.crs}")
    print(f"  {'Resolution':<14} {src.res[0]:.3f}, {src.res[1]:.3f}")
    print(f"  {'Bounds':<14} {src.bounds}")
    print(f"  {'NoData':<14} {src.nodata}")

Visual RGB pre-event
--------------------
  Size (W × H)   17,408 × 33,792 px
  Bands          3
  Data type      uint8
  CRS            EPSG:32735
  Resolution     0.305, 0.305
  Bounds         BoundingBox(left=709843.75, bottom=9774843.75, right=715156.25, top=9785156.25)
  NoData         None

Visual RGB post-event
---------------------
  Size (W × H)   17,408 × 33,792 px
  Bands          3
  Data type      uint8
  CRS            EPSG:32735
  Resolution     0.305, 0.305
  Bounds         BoundingBox(left=709843.75, bottom=9774843.75, right=715156.25, top=9785156.25)
  NoData         None


### 2.2 Buildings data

To let the train it needs to know where buildings stand. For the area at Lake Kivu there is no dataset which provides the outlines of the buildings. Google's Earth Engine provides a data set with buildings, but unfortunately the confidence level are low. Therefore, the building used to train the model are hand drawn by the author in QGIS.

In [ ]:
# read in the buildings
buildings = gpd.read_file("../GIS/buildings.gpkg")

---
## Data processing

To reduce runtime the the total area where the model later should run needs to be decreased. Also training and validation area need to be defined.


### 3.1 Select Area of Interest (AOI)

In this chapter the raster files get clipped into an AOI for the area that should be analysed for the damage. In addition, we clip AOI's for training and validation are selected. The training and the validation happen on the post-event image. The training area is approximately five times bigger than the Validation area. The model is later applied on the whole pre- and also post-event image.\
The AOI for the total AOI, training and validation set are drawn as polygons in QGIS as a GeoPackage file.

In [19]:
# reduce the size of the total tif area
tif_names = ["rgb_post.tif", "rgb_pre.tif"]

clip_tifs(
    tif_names,
    aoi_path="../GIS/aoi_total_area.gpkg",
    folder="../data/processed",
    suffix="_clipped"
)

clipping rgb_post.tif
clipping rgb_pre.tif


[WindowsPath('../data/processed/rgb_post_clipped.tif'),
 WindowsPath('../data/processed/rgb_pre_clipped.tif')]

In [24]:
tif_names = ["rgb_post_clipped.tif", "rgb_pre_clipped.tif"]

# clip training data aoi
clip_tifs(
    tif_names,
    aoi_path="../GIS/aoi_training.gpkg",
    folder="../data/processed",
    suffix="_training"
)

# clip validation data aoi
clip_tifs(
    tif_names,
    aoi_path="../GIS/aoi_validation.gpkg",
    folder="../data/processed",
    suffix="_validation"
)

clipping rgb_post_clipped.tif
clipping rgb_pre_clipped.tif
clipping rgb_post_clipped.tif
clipping rgb_pre_clipped.tif


[WindowsPath('../data/processed/rgb_post_clipped_validation.tif'),
 WindowsPath('../data/processed/rgb_pre_clipped_validation.tif')]

## 4. Train the model

In this chapter the model to detect buildings on the images is trained. 